# Быстрая проверка решения

Используется готовая модель с 38 признаками, но история, поиск и признаки для всех 2 452 запросов рассчитываются заново. Итог сравнивается с приложенным `answer.csv` побайтово.
Положите три исходных Parquet в `data/`, выберите ядро **Avito ranking** и нажмите **Run → Run All Cells**. Установка описана в [README](../README.md).
`config_path` указывает на настройки путей в `configs/improved.toml`. Вызов `run_stage` запускает этап через `subprocess` в отдельном процессе: память освобождается после завершения, полный вывод остаётся в `results/improved/logs/`.
Первый кодовый блок загружает настройки и запускает тесты. Обучение модели в этом ноутбуке не повторяется. Подробная [схема расчёта](../docs/pipeline.md) объясняет связь этапов и модулей.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
import time

# Ограничиваем число потоков для работы на CPU с небольшим объёмом памяти.
os.environ.update({"POLARS_MAX_THREADS": "2", "OPENBLAS_NUM_THREADS": "2", "OMP_NUM_THREADS": "2"})

from avito_ranker.config import load_config
from avito_improved.run import run_stage

# Пути работают при запуске из корня репозитория и из папки notebooks.
project_dir = Path.cwd().resolve()
if project_dir.name == "notebooks":
    # Пути работают при запуске из корня репозитория и из папки notebooks.
    project_dir = project_dir.parent
# Оба ноутбука используют одни настройки и одни модули расчёта.
config_path = project_dir / "configs/improved.toml"
config = load_config(config_path)
work = config["work_dir"] / "quality"
saved = config["results_dir"]
started = time.perf_counter()
# Сначала запускаем тесты; при ошибке выполнение остановится.
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
               cwd=project_dir, check=True)

## 1. Подготовка

Из исходных данных восстанавливаются разбиение, история обучающих запросов и центры локаций. Затем готовая модель из `results/improved/` копируется в `work/quality/`.
Код, модель, данные и история сравниваются с сохранённым снимком эксперимента. При несовпадении выполнение останавливается.
Результат: `work/quality/history/` и `work/quality/ranker.cbm`; в логе должно появиться «Модель и контрольные суммы проверены».

In [ ]:
# Готовим разбиение, корпус и историю только по обучающим группам.
run_stage("prepare", config_path)
# Загружаем готовую модель и сверяем код, данные и историю со снимком.
run_stage("restore", config_path)

## 2. Поисковые индексы

Для 189 212 объявлений из `benchmark_items.parquet` строятся отдельные BM25-индексы заголовков, описаний и параметров. Объявления только из train в этот корпус не входят.
Результат: три индекса в `work/benchmark_indices/`. Успешное завершение всех трёх расчётов подтверждается в выводе этапа; подробности сохраняются в логах.

In [ ]:
# Индексируем только объявления, допустимые в итоговом ответе.
run_stage("benchmark_indices", config_path)

## 3. Кандидаты и признаки

Для каждого benchmark-запроса объединяются кандидаты BM25, поиска по фильтрам, обучающей истории и близким локациям. Для каждой пары рассчитываются 38 признаков без проверочной разметки.
Результат: `work/quality/benchmark.parquet` и сводка `benchmark_candidates.json` в той же папке. В сводке должно быть 2 452 запроса.
На этом этапе модель ещё не выбирает итоговые 50 объявлений.

In [ ]:
# Повторяем поиск и расчёт признаков для всех запросов benchmark.
run_stage("benchmark_features", config_path)

## 4. Ответ

Готовая модель оценивает кандидатов. Сохранённый способ объединения берёт первые 40 её объявлений и дополняет их исходной выдачей, пропуская повторы.
Внутренние номера документов заменяются исходными `item_id`. Создаются `work/quality/answer.csv` и `work/quality/submission_check.json`.
Успех: `valid: true`, 2 452 строки и по 50 уникальных допустимых объявлений в каждой.

In [ ]:
# Применяем сохранённую модель и записываем проверенный CSV.
run_stage("predict", config_path)

## 5. Сравнение с готовым файлом

Повторно проверяются колонки, query_id и item_id, затем рассчитанный `work/quality/answer.csv` сравнивается с `answer.csv` в корне репозитория. Сравнение побайтовое, включая окончания строк.
Только при совпадении в `work/quality/check_report.json` записываются `status: passed` и `answer_identical: true`. Последняя ячейка показывает этот отчёт и время проверки.

In [ ]:
from avito_retrieval.submission import validate_answer

# Проверяем формат, полный список запросов и допустимость всех item_id.
computed = work / "answer.csv"
reference = project_dir / "answer.csv"
report = validate_answer(computed, config["data_dir"] / "benchmark_queries.parquet",
                         config["data_dir"] / "benchmark_items.parquet")
# Требуем точного совпадения файлов, включая порядок и переводы строк.
if computed.read_bytes() != reference.read_bytes():
    raise AssertionError("Новый answer.csv отличается от приложенного")
# Успешный отчёт записывается только после всех проверок.
report.update({"status": "passed", "answer_identical": True,
               "seconds": round(time.perf_counter() - started, 2)})
(work / "check_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
report